# 04 - Trimodal Inference Pipeline

This notebook runs end-to-end inference using the trained `TrimodalFusionModel`.

**Pipeline**:
1. Load pre-extracted features from `data/raw/<PID>/` (auto-detected CSV files)
2. Normalize per-modality features
3. Run `TrimodalFusionModel` (Audio + Video + Text)
4. Output PHQ-8 score / depression probability, severity, modality contributions

**Supports both modes**:
- **Regression** (num_classes=1): PHQ-8 score (0-24) + severity label
- **Classification** (num_classes=2): depression probability + predicted label (0/1)

**Expected files per participant** in `data/raw/<PID>/`:
- `<PID>_OpenSMILE2.3.0_egemaps.csv` (88-dim)
- `<PID>_OpenSMILE2.3.0_mfcc.csv` (39 or 120-dim)
- `<PID>_BoVW_openFace_2.1.0_Pose_Gaze_AUs.csv` (49-dim)
- `<PID>_CNN_ResNet.csv` (512-dim)
- `<PID>_Transcript.csv` (text utterances)

Supports graceful degradation: if a modality is missing, the model uses learned tokens.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.inference.trimodal_pipeline import TrimodalPredictor, TrimodalPredictionResult

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 4.1 Load trained model

In [ ]:
CHECKPOINT_PATH = "../checkpoints/best_trimodal.pt"
# For classification mode, use: CHECKPOINT_PATH = "../checkpoints/best_classifier.pt"

# Auto-detect num_classes from checkpoint if available
num_classes = 1
try:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    cfg = ckpt.get("config", {})
    num_classes = cfg.get("model", {}).get("num_classes", 1)
    print(f"Auto-detected num_classes={num_classes} from checkpoint config")
except Exception:
    print("Could not auto-detect num_classes, defaulting to 1 (regression)")

predictor = TrimodalPredictor(
    checkpoint_path=CHECKPOINT_PATH,
    device=str(device),
    fusion_dim=128,
    openface_dim=49,
    cnn_dim=512,
    num_classes=num_classes,
)

print(f"TrimodalPredictor loaded. Mode: {'classification' if num_classes > 1 else 'regression'}")

## 4.2 Single patient inference from organized folder

In [ ]:
RAW_DIR = Path("../data/raw")
sample_pid = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])[0]
patient_folder = RAW_DIR / sample_pid

print(f"Running inference for: {sample_pid}")
print(f"Folder: {patient_folder}")

result = predictor.predict_from_patient_folder(
    patient_folder=patient_folder,
    participant_id=sample_pid,
    debug=True,
)

print(f"\n=== Prediction Result ===")
print(f"  PHQ-8 Score:       {result.phq8_score}")
print(f"  Severity:          {result.severity}")
print(f"  Confidence:        {result.confidence}")
print(f"  Modalities Used:   {result.modalities_used}")
if result.is_classification:
    print(f"  Mode:              CLASSIFICATION")
    print(f"  Depression Prob:   {result.depression_probability:.4f}")
    print(f"  Predicted Label:   {result.predicted_label} ({'depressed' if result.predicted_label == 1 else 'non-depressed'})")
else:
    print(f"  Mode:              REGRESSION")
print(f"  Modality Contributions:")
for mod, contrib in result.modality_contributions.items():
    print(f"    {mod}: {contrib:.3f}")
print(f"  Inference Time:    {result.inference_time_s:.3f}s")

if result.debug:
    print(f"\n  Debug Info:")
    for k, v in result.debug.items():
        print(f"    {k}: {v}")

## 4.3 Batch inference across all patients

In [ ]:
from tqdm.notebook import tqdm

results = []
failed = []

for patient_dir in tqdm(sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])):
    pid = patient_dir.name
    try:
        result = predictor.predict_from_patient_folder(patient_dir, participant_id=pid)
        row = {
            'patient_id': pid,
            'phq8_score': result.phq8_score,
            'severity': result.severity,
            'confidence': result.confidence,
            'modalities_used': ','.join(result.modalities_used),
            'audio_contrib': result.modality_contributions.get('audio', 0),
            'video_contrib': result.modality_contributions.get('video', 0),
            'text_contrib': result.modality_contributions.get('text', 0),
            'inference_time': result.inference_time_s,
            'is_classification': result.is_classification,
        }
        if result.is_classification:
            row['depression_probability'] = result.depression_probability
            row['predicted_label'] = result.predicted_label
        results.append(row)
    except Exception as e:
        failed.append((pid, str(e)))

results_df = pd.DataFrame(results)
print(f"\nSuccessful: {len(results_df)} patients")
print(f"Failed: {len(failed)} patients")

if not results_df.empty:
    print(f"\nPHQ-8 Score Distribution:")
    print(results_df['phq8_score'].describe())
    print(f"\nSeverity Counts:")
    print(results_df['severity'].value_counts())
    if results_df['is_classification'].any():
        print(f"\nClassification Results:")
        print(results_df['predicted_label'].value_counts())
    display(results_df.head(10))

## 4.4 Visualize modality contributions

In [ ]:
import matplotlib.pyplot as plt

if not results_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for ax, col, title in zip(
        axes,
        ['audio_contrib', 'video_contrib', 'text_contrib'],
        ['Audio Contribution', 'Video Contribution', 'Text Contribution']
    ):
        ax.hist(results_df[col], bins=20, edgecolor='black')
        ax.set_title(title)
        ax.set_xlabel('Contribution Weight')
        ax.set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()
    
    # Scatter: Audio vs Video contribution
    fig, ax = plt.subplots(figsize=(6, 6))
    scatter = ax.scatter(
        results_df['audio_contrib'],
        results_df['video_contrib'],
        c=results_df['phq8_score'],
        cmap='RdYlGn_r',
        alpha=0.7
    )
    ax.set_xlabel('Audio Contribution')
    ax.set_ylabel('Video Contribution')
    ax.set_title('Audio vs Video Contribution (color = PHQ-8)')
    plt.colorbar(scatter, label='PHQ-8 Score')
    plt.show()

## 4.5 Save predictions

In [ ]:
output_csv = Path("../outputs/trimodal_predictions.csv")
output_csv.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_csv, index=False)
print(f"Predictions saved to: {output_csv}")

# Also save failed list for debugging
if failed:
    failed_df = pd.DataFrame(failed, columns=['patient_id', 'error'])
    failed_df.to_csv(output_csv.parent / "failed_patients.csv", index=False)
    print(f"Failed list saved to: {output_csv.parent / 'failed_patients.csv'}")

## 4.6 Manual feature input inference (advanced)

For cases where you have features in memory (e.g., from real-time extraction).

In [ ]:
# Example: create dummy features for demonstration
dummy_audio = np.random.randn(10, 208).astype(np.float32)   # 10 chunks, 208 dim
dummy_video_openface = np.random.randn(50, 49).astype(np.float32)  # 50 frames, 49 dim
dummy_video_cnn = np.random.randn(50, 512).astype(np.float32)      # 50 frames, 512 dim
dummy_text = np.random.randn(8, 384).astype(np.float32)            # 8 utterances, 384 dim

manual_result = predictor.predict(
    audio_features=dummy_audio,
    video_openface=dummy_video_openface,
    video_cnn=dummy_video_cnn,
    text_features=dummy_text,
    participant_id="DEMO_001",
)

print(f"Manual inference result:")
print(f"  Score: {manual_result.phq8_score}")
print(f"  Severity: {manual_result.severity}")
print(f"  Modalities: {manual_result.modalities_used}")
print(f"  Contributions: {manual_result.modality_contributions}")